<a href="https://colab.research.google.com/github/siddhimishra20/compiler-construction-lab/blob/main/week2/week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installations

In [ ]:
# 1. Install flex and bison
!apt-get install -y flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 1,072 kB in 1s (828 kB/s)
Selecting previously unselected package flex.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpa

In [ ]:
%%writefile compi.l
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "compi.tab.h"

int line = 1;
%}

DIGIT      [0-9]
LETTER     [A-Za-z_]
ID         {LETTER}({LETTER}|{DIGIT})*
ICONST     0|[1-9]{DIGIT}*
FCONST     {DIGIT}+"."{DIGIT}+

%%

"int"        { return INT; }
"float"      { return FLOAT; }
"if"         { return IF; }
"else"       { return ELSE; }
"while"      { return WHILE; }
"print"      { return PRINT; }

{FCONST}     { yylval.fval = atof(yytext); return FCONST; }
{ICONST}     { yylval.ival = atoi(yytext); return ICONST; }
{ID}         { yylval.sval = strdup(yytext); return ID; }

"=="         { return EQ; }
"!="         { return NE; }
"<="         { return LE; }
">="         { return GE; }
"&&"         { return AND; }
"||"         { return OR; }

"<"          { return '<'; }
">"          { return '>'; }
"="          { return '='; }

"+"          { return '+'; }
"-"          { return '-'; }
"*"          { return '*'; }
"/"          { return '/'; }
"%"          { return '%'; }
"!"          { return '!'; }

"("          { return '('; }
")"          { return ')'; }
"{"          { return '{'; }
"}"          { return '}'; }
";"          { return ';'; }
","          { return ','; }

[ \t]+       { }
\n           { line++; }

.            { printf("Lexical error at line %d: %s\n", line, yytext); }

%%

int yywrap() { return 1; }

Writing compi.l


In [ ]:
%%writefile compi.y
%{
#include <stdio.h>
#include <stdlib.h>

extern int line;
int yylex(void);
void yyerror(char *s);
%}

%union {
    int ival;
    float fval;
    char* sval;
}

%token <sval> ID
%token <ival> ICONST
%token <fval> FCONST
%token INT FLOAT IF ELSE WHILE PRINT
%token EQ NE LE GE AND OR

/* Operator Precedence and Associativity */
%left OR
%left AND
%left EQ NE LE GE '<' '>'
%left '+' '-'
%left '*' '/' '%'
%right '!'
%nonassoc LOWER_THAN_ELSE
%nonassoc ELSE

%%

/* The program is a series of units (declarations or statements) */
program:
    unit_list
    ;

unit_list:
    unit_list unit
    | /* empty */
    ;

unit:
    declaration
    | statement
    ;

declaration:
    type ID ';' { printf("Line %d: Syntactic Validation [Declaration: %s]\n", line, $2); free($2); }
    ;

type:
    INT | FLOAT
    ;

statement:
    assignment_stmt
    | if_stmt
    | while_stmt
    | print_stmt
    | compound_stmt
    ;

/* Allows declarations and statements to mix inside braces */
compound_stmt:
    '{' unit_list '}'
    ;

assignment_stmt:
    ID '=' expression ';' { printf("Line %d: Syntactic Validation [Assignment to %s]\n", line, $1); free($1); }
    ;

if_stmt:
    IF '(' condition ')' statement %prec LOWER_THAN_ELSE
    | IF '(' condition ')' statement ELSE statement { printf("Line %d: Syntactic Validation [If-Else Block]\n", line); }
    ;

while_stmt:
    WHILE '(' condition ')' statement { printf("Line %d: Syntactic Validation [While Loop]\n", line); }
    ;

print_stmt:
    PRINT '(' expression ')' ';' { printf("Line %d: Syntactic Validation [Print Statement]\n", line); }
    ;

expression:
    expression '+' expression
    | expression '-' expression
    | expression '*' expression
    | expression '/' expression
    | expression '%' expression
    | '(' expression ')'
    | ID { free($1); }
    | ICONST
    | FCONST
    ;

condition:
    expression EQ expression
    | expression NE expression
    | expression '<' expression
    | expression '>' expression
    | expression LE expression
    | expression GE expression
    | '(' condition ')'
    | '!' condition
    | condition AND condition
    | condition OR condition
    ;

%%

int main() {
    if (yyparse() == 0) {
        printf("\nRESULT: Syntactic Validation Successful.\n");
    } else {
        printf("\nRESULT: Syntactic Validation Failed.\n");
    }
    return 0;
}

void yyerror(char *s) {
    fprintf(stderr, "Syntax Error at line %d: %s\n", line, s);
}

Writing compi.y


In [ ]:
%%writefile program.txt

int a;
int b;
int sum;
float avg;
a = 2 * (3 + 4);
b = 15;
sum = 0;
while (a < b && b != 0) {
 int temp;
 temp = a * 2;
 if ((temp % 3 == 0) || (a > 5)) {
 sum = sum + temp;
 } else {
 sum = sum - 1;
 }
 a = a + 1;
}
avg = sum / (b - a);
if (!(avg < 5.0)) {
 print(sum);
} else {
 print(avg);
}


Writing program.txt


In [ ]:
# 2. Generate Parser and Lexer files
!bison -d compi.y
!flex compi.l

# 3. Compile the C files
!gcc compi.tab.c lex.yy.c -o compiler -lfl

# 4. Run the program
!./compiler < program.txt

Line 2: Syntactic Validation [Declaration: a]
Line 3: Syntactic Validation [Declaration: b]
Line 4: Syntactic Validation [Declaration: sum]
Line 5: Syntactic Validation [Declaration: avg]
Line 6: Syntactic Validation [Assignment to a]
Line 7: Syntactic Validation [Assignment to b]
Line 8: Syntactic Validation [Assignment to sum]
Line 10: Syntactic Validation [Declaration: temp]
Line 11: Syntactic Validation [Assignment to temp]
Line 13: Syntactic Validation [Assignment to sum]
Line 15: Syntactic Validation [Assignment to sum]
Line 16: Syntactic Validation [If-Else Block]
Line 17: Syntactic Validation [Assignment to a]
Line 18: Syntactic Validation [While Loop]
Line 19: Syntactic Validation [Assignment to avg]
Line 21: Syntactic Validation [Print Statement]
Line 23: Syntactic Validation [Print Statement]
Line 24: Syntactic Validation [If-Else Block]

RESULT: Syntactic Validation Successful.


In [ ]:
%%writefile compi.y

%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

extern int line;
int yylex(void);
void yyerror(char *s);

/* ─── Parse Tree Node ─────────────────────────────────────────────── */
typedef struct Node {
    char label[256];
    struct Node *children[16];
    int nchildren;
} Node;

Node *make_node(const char *label) {
    Node *n = calloc(1, sizeof(Node));
    strncpy(n->label, label, 255);
    return n;
}

Node *add_child(Node *parent, Node *child) {
    if (parent && child && parent->nchildren < 16)
        parent->children[parent->nchildren++] = child;
    return parent;
}

Node *make_leaf(const char *label) {
    return make_node(label);
}

/* ─── Root of the whole program tree ─────────────────────────────── */
Node *program_root = NULL;

/* ─── Pick the "most interesting" statement ──────────────────────── */
Node *chosen_stmt = NULL;   /* the subtree we'll analyse */
char chosen_text[512] = ""; /* its source text */

int stmt_complexity(Node *n) {
    /* rough heuristic: count total nodes */
    if (!n) return 0;
    int s = 1;
    for (int i = 0; i < n->nchildren; i++) s += stmt_complexity(n->children[i]);
    return s;
}

void try_pick(Node *n, const char *src) {
    if (!n) return;
    int c = stmt_complexity(n);
    if (c > stmt_complexity(chosen_stmt)) {
        chosen_stmt = n;
        strncpy(chosen_text, src, 511);
    }
}

/* ─── ASCII tree printer ──────────────────────────────────────────── */
void print_tree(Node *n, const char *prefix, int is_last) {
    if (!n) return;
    printf("%s%s%s\n", prefix, is_last ? "└── " : "├── ", n->label);
    char new_prefix[1024];
    snprintf(new_prefix, sizeof(new_prefix), "%s%s", prefix, is_last ? "    " : "│   ");
    for (int i = 0; i < n->nchildren; i++)
        print_tree(n->children[i], new_prefix, i == n->nchildren - 1);
}

/* ─── Derivation helpers ─────────────────────────────────────────── */
/* We store each derivation step as a string and print them at the end */
#define MAX_STEPS 128
char *lmd_steps[MAX_STEPS];
int   lmd_count = 0;
char *rmd_steps[MAX_STEPS];
int   rmd_count = 0;

void lmd_push(const char *s) {
    if (lmd_count < MAX_STEPS) lmd_steps[lmd_count++] = strdup(s);
}
void rmd_push(const char *s) {
    if (rmd_count < MAX_STEPS) rmd_steps[rmd_count++] = strdup(s);
}

/* Build LMD steps from the chosen tree (left-to-right expansion) */
/* We produce one step per internal node, replacing its label with its
   children's labels, working strictly left-to-right (LMD). */

/* Serialise a subtree to a flat string of labels */
void flatten(Node *n, char *buf, int bufsz) {
    if (!n) return;
    if (n->nchildren == 0) {
        /* leaf */
        strncat(buf, n->label, bufsz - strlen(buf) - 1);
        strncat(buf, " ", bufsz - strlen(buf) - 1);
    } else {
        strncat(buf, "<", bufsz - strlen(buf) - 1);
        strncat(buf, n->label, bufsz - strlen(buf) - 1);
        strncat(buf, "> ", bufsz - strlen(buf) - 1);
    }
}

/* Serialise the *whole* tree (depth-first left-to-right), but for
   a "sentential form" we show non-terminals as <X> and terminals bare */
void sentential(Node *n, char *buf, int bufsz) {
    if (!n) return;
    if (n->nchildren == 0) {
        strncat(buf, n->label, bufsz - strlen(buf) - 1);
        strncat(buf, " ", bufsz - strlen(buf) - 1);
    } else {
        for (int i = 0; i < n->nchildren; i++)
            sentential(n->children[i], buf, bufsz);
    }
}

/* Generate LMD steps by walking the tree and recording each expansion */
void gen_lmd(Node *n, char *current_form, int form_size) {
    if (!n || n->nchildren == 0) return;
    /* Find "<label>" in current_form and replace with children */
    char search[300]; snprintf(search, sizeof(search), "<%s>", n->label);
    char *pos = strstr(current_form, search);
    if (!pos) return;

    /* Build replacement string from children */
    char replacement[2048] = "";
    for (int i = 0; i < n->nchildren; i++) {
        if (n->children[i]->nchildren == 0) {
            strncat(replacement, n->children[i]->label, sizeof(replacement)-strlen(replacement)-1);
        } else {
            strncat(replacement, "<", sizeof(replacement)-strlen(replacement)-1);
            strncat(replacement, n->children[i]->label, sizeof(replacement)-strlen(replacement)-1);
            strncat(replacement, ">", sizeof(replacement)-strlen(replacement)-1);
        }
        strncat(replacement, " ", sizeof(replacement)-strlen(replacement)-1);
    }

    /* Build new form */
    char new_form[4096] = "";
    int before = pos - current_form;
    strncat(new_form, current_form, before);
    strncat(new_form, replacement, sizeof(new_form)-strlen(new_form)-1);
    strncat(new_form, pos + strlen(search), sizeof(new_form)-strlen(new_form)-1);

    lmd_push(new_form);
    strncpy(current_form, new_form, form_size - 1);

    /* Recurse into children left-to-right */
    for (int i = 0; i < n->nchildren; i++)
        gen_lmd(n->children[i], current_form, form_size);
}

/* Generate RMD by walking rightmost-first */
void gen_rmd(Node *n, char *current_form, int form_size) {
    if (!n || n->nchildren == 0) return;
    char search[300]; snprintf(search, sizeof(search), "<%s>", n->label);
    char *pos = strstr(current_form, search);
    if (!pos) return;

    char replacement[2048] = "";
    for (int i = 0; i < n->nchildren; i++) {
        if (n->children[i]->nchildren == 0) {
            strncat(replacement, n->children[i]->label, sizeof(replacement)-strlen(replacement)-1);
        } else {
            strncat(replacement, "<", sizeof(replacement)-strlen(replacement)-1);
            strncat(replacement, n->children[i]->label, sizeof(replacement)-strlen(replacement)-1);
            strncat(replacement, ">", sizeof(replacement)-strlen(replacement)-1);
        }
        strncat(replacement, " ", sizeof(replacement)-strlen(replacement)-1);
    }

    char new_form[4096] = "";
    int before = pos - current_form;
    strncat(new_form, current_form, before);
    strncat(new_form, replacement, sizeof(new_form)-strlen(new_form)-1);
    strncat(new_form, pos + strlen(search), sizeof(new_form)-strlen(new_form)-1);

    rmd_push(new_form);
    strncpy(current_form, new_form, form_size - 1);

    /* Recurse into children right-to-left for RMD */
    for (int i = n->nchildren - 1; i >= 0; i--)
        gen_rmd(n->children[i], current_form, form_size);
}

void print_derivation(char **steps, int count, const char *title, const char *abbrev) {
    printf("\n%s\n", title);
    for (int i = 0; i < 60; i++) putchar('-');
    printf("\nStatement: %s\n\n", chosen_text);
    for (int i = 0; i < count; i++) {
        if (i == 0)
            printf("  <statement>\n");
        printf("    => %s\n", steps[i]);
    }
}

%}

%union {
    int ival;
    float fval;
    char* sval;
    struct Node* node;
}

%token <sval> ID
%token <ival> ICONST
%token <fval> FCONST
%token INT FLOAT IF ELSE WHILE PRINT
%token EQ NE LE GE AND OR

%left OR
%left AND
%left EQ NE LE GE '<' '>'
%left '+' '-'
%left '*' '/' '%'
%right '!'
%nonassoc LOWER_THAN_ELSE
%nonassoc ELSE

%type <node> program unit_list unit declaration type statement
%type <node> compound_stmt assignment_stmt if_stmt while_stmt print_stmt
%type <node> expression condition

%%

program:
    unit_list {
        Node *n = make_node("program");
        add_child(n, $1);
        program_root = n;
        $$ = n;
    }
    ;

unit_list:
    unit_list unit {
        add_child($1, $2);
        $$ = $1;
    }
    | /* empty */ {
        $$ = make_node("unit_list");
    }
    ;

unit:
    declaration { $$ = $1; }
    | statement { $$ = $1; }
    ;

declaration:
    type ID ';' {
        printf("Line %d: Syntactic Validation [Declaration: %s]\n", line, $2);
        Node *n = make_node("declaration");
        add_child(n, $1);
        char tmp[256]; snprintf(tmp, sizeof(tmp), "%s", $2);
        add_child(n, make_leaf(tmp));
        add_child(n, make_leaf(";"));
        free($2);
        $$ = n;
    }
    ;

type:
    INT   { $$ = make_leaf("INT"); }
    | FLOAT { $$ = make_leaf("FLOAT"); }
    ;

statement:
    assignment_stmt { $$ = $1; }
    | if_stmt       { $$ = $1; }
    | while_stmt    { $$ = $1; }
    | print_stmt    { $$ = $1; }
    | compound_stmt { $$ = $1; }
    ;

compound_stmt:
    '{' unit_list '}' {
        Node *n = make_node("compound_stmt");
        add_child(n, make_leaf("{"));
        add_child(n, $2);
        add_child(n, make_leaf("}"));
        $$ = n;
    }
    ;

assignment_stmt:
    ID '=' expression ';' {
        printf("Line %d: Syntactic Validation [Assignment to %s]\n", line, $1);
        Node *n = make_node("assignment_stmt");
        char tmp[256]; snprintf(tmp, sizeof(tmp), "%s", $1);
        add_child(n, make_leaf(tmp));
        add_child(n, make_leaf("="));
        add_child(n, $3);
        add_child(n, make_leaf(";"));
        free($1);
        /* track source text for derivation */
        char src[512]; snprintf(src, sizeof(src), "%s = <expr> ;", tmp);
        try_pick(n, src);
        $$ = n;
    }
    ;

if_stmt:
    IF '(' condition ')' statement %prec LOWER_THAN_ELSE {
        Node *n = make_node("if_stmt");
        add_child(n, make_leaf("IF"));
        add_child(n, make_leaf("("));
        add_child(n, $3);
        add_child(n, make_leaf(")"));
        add_child(n, $5);
        try_pick(n, "IF ( <condition> ) <statement>");
        $$ = n;
    }
    | IF '(' condition ')' statement ELSE statement {
        printf("Line %d: Syntactic Validation [If-Else Block]\n", line);
        Node *n = make_node("if_stmt");
        add_child(n, make_leaf("IF"));
        add_child(n, make_leaf("("));
        add_child(n, $3);
        add_child(n, make_leaf(")"));
        add_child(n, $5);
        add_child(n, make_leaf("ELSE"));
        add_child(n, $7);
        try_pick(n, "IF ( <condition> ) <statement> ELSE <statement>");
        $$ = n;
    }
    ;

while_stmt:
    WHILE '(' condition ')' statement {
        printf("Line %d: Syntactic Validation [While Loop]\n", line);
        Node *n = make_node("while_stmt");
        add_child(n, make_leaf("WHILE"));
        add_child(n, make_leaf("("));
        add_child(n, $3);
        add_child(n, make_leaf(")"));
        add_child(n, $5);
        try_pick(n, "WHILE ( <condition> ) <statement>");
        $$ = n;
    }
    ;

print_stmt:
    PRINT '(' expression ')' ';' {
        printf("Line %d: Syntactic Validation [Print Statement]\n", line);
        Node *n = make_node("print_stmt");
        add_child(n, make_leaf("PRINT"));
        add_child(n, make_leaf("("));
        add_child(n, $3);
        add_child(n, make_leaf(")"));
        add_child(n, make_leaf(";"));
        try_pick(n, "PRINT ( <expression> ) ;");
        $$ = n;
    }
    ;

expression:
    expression '+' expression {
        Node *n = make_node("expression");
        add_child(n, $1); add_child(n, make_leaf("+")); add_child(n, $3);
        $$ = n;
    }
    | expression '-' expression {
        Node *n = make_node("expression");
        add_child(n, $1); add_child(n, make_leaf("-")); add_child(n, $3);
        $$ = n;
    }
    | expression '*' expression {
        Node *n = make_node("expression");
        add_child(n, $1); add_child(n, make_leaf("*")); add_child(n, $3);
        $$ = n;
    }
    | expression '/' expression {
        Node *n = make_node("expression");
        add_child(n, $1); add_child(n, make_leaf("/")); add_child(n, $3);
        $$ = n;
    }
    | expression '%' expression {
        Node *n = make_node("expression");
        add_child(n, $1); add_child(n, make_leaf("%")); add_child(n, $3);
        $$ = n;
    }
    | '(' expression ')' {
        Node *n = make_node("expression");
        add_child(n, make_leaf("(")); add_child(n, $2); add_child(n, make_leaf(")"));
        $$ = n;
    }
    | ID {
        Node *n = make_node("expression");
        char tmp[256]; snprintf(tmp, sizeof(tmp), "%s", $1);
        add_child(n, make_leaf(tmp));
        free($1);
        $$ = n;
    }
    | ICONST {
        Node *n = make_node("expression");
        char tmp[64]; snprintf(tmp, sizeof(tmp), "%d", $1);
        add_child(n, make_leaf(tmp));
        $$ = n;
    }
    | FCONST {
        Node *n = make_node("expression");
        char tmp[64]; snprintf(tmp, sizeof(tmp), "%g", $1);
        add_child(n, make_leaf(tmp));
        $$ = n;
    }
    ;

condition:
    expression EQ expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("==")); add_child(n, $3);
        $$ = n;
    }
    | expression NE expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("!=")); add_child(n, $3);
        $$ = n;
    }
    | expression '<' expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("<")); add_child(n, $3);
        $$ = n;
    }
    | expression '>' expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf(">")); add_child(n, $3);
        $$ = n;
    }
    | expression LE expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("<=")); add_child(n, $3);
        $$ = n;
    }
    | expression GE expression {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf(">=")); add_child(n, $3);
        $$ = n;
    }
    | '(' condition ')' {
        Node *n = make_node("condition");
        add_child(n, make_leaf("(")); add_child(n, $2); add_child(n, make_leaf(")"));
        $$ = n;
    }
    | '!' condition {
        Node *n = make_node("condition");
        add_child(n, make_leaf("!")); add_child(n, $2);
        $$ = n;
    }
    | condition AND condition {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("&&")); add_child(n, $3);
        $$ = n;
    }
    | condition OR condition {
        Node *n = make_node("condition");
        add_child(n, $1); add_child(n, make_leaf("||")); add_child(n, $3);
        $$ = n;
    }
    ;

%%

/* ── Free tree memory ─────────────────────────────────────────────── */
void free_tree(Node *n) {
    if (!n) return;
    for (int i = 0; i < n->nchildren; i++) free_tree(n->children[i]);
    free(n);
}

int main() {
    if (yyparse() != 0) {
        printf("\nRESULT: Syntactic Validation Failed.\n");
        return 1;
    }
    printf("\nRESULT: Syntactic Validation Successful.\n");

    if (!chosen_stmt) {
        printf("(No statement selected for derivation analysis.)\n");
        return 0;
    }

    /* ── Separator ─────────────────────────────────────────────────── */
    printf("\n");
    for (int i = 0; i < 60; i++) putchar('=');
    printf("\n  POST-PARSE ANALYSIS  (chosen statement)\n");
    for (int i = 0; i < 60; i++) putchar('=');

    /* ── Leftmost Derivation ───────────────────────────────────────── */
    char lmd_form[4096] = "";
    snprintf(lmd_form, sizeof(lmd_form), "<%s>", chosen_stmt->label);
    gen_lmd(chosen_stmt, lmd_form, sizeof(lmd_form));
    print_derivation(lmd_steps, lmd_count,
                     "\n1. LEFTMOST DERIVATION (LMD)", "LMD");

    /* ── Rightmost Derivation ─────────────────────────────────────── */
    char rmd_form[4096] = "";
    snprintf(rmd_form, sizeof(rmd_form), "<%s>", chosen_stmt->label);
    gen_rmd(chosen_stmt, rmd_form, sizeof(rmd_form));
    print_derivation(rmd_steps, rmd_count,
                     "\n2. RIGHTMOST DERIVATION (RMD)", "RMD");

    /* ── Visual Syntax Tree ───────────────────────────────────────── */
    printf("\n");
    for (int i = 0; i < 60; i++) putchar('=');
    printf("\n3. VISUAL SYNTAX TREE\n");
    for (int i = 0; i < 60; i++) putchar('-');
    printf("\nStatement: %s\n\n", chosen_text);
    print_tree(chosen_stmt, "", 1);

    /* cleanup */
    for (int i = 0; i < lmd_count; i++) free(lmd_steps[i]);
    for (int i = 0; i < rmd_count; i++) free(rmd_steps[i]);
    free_tree(program_root);
    return 0;
}

void yyerror(char *s) {
    fprintf(stderr, "Syntax Error at line %d: %s\n", line, s);
}


Overwriting compi.y
